# Trabajo Práctico Grupal: flujo de procesamiento de datos
## "Pipeline de control de presión arterial"

- **Duración:** 45 minutos de programación + 15 minutos de armado del repositorio
- **Modalidad:** grupos de 3 o 4 integrantes
- **Entrega:** repositorio público en GitHub (el enlace se entrega al finalizar la clase)

---

### Propósito
Esta actividad es una versión reducida del **Trabajo Práctico 2**. Se recorre una vez, de forma guiada, el flujo completo que el TP2 exige:

**leer → inspeccionar y validar → limpiar y transformar → analizar → comunicar el resultado**

**Problema.** Un equipo de salud recibe las mediciones de presión arterial de un estudio en dos formatos, CSV y JSON. Los archivos contienen errores de carga: códigos mal escritos, registros duplicados y valores faltantes, no numéricos o fuera de rango. Se necesita un programa que descarte esos registros dejando constancia del motivo y que produzca un resumen por grupo del estudio.

### Relación con el TP2
| Parte | Contenido | Paso del TP2 |
|---|---|---|
| 0 | Archivos de datos | 2. Datos y procedencia |
| 1 | `FuenteDatos`, `FuenteCSV`, `FuenteJSON` | 4. Abstracción común y dos implementaciones |
| 2 | Clase `Paciente` | 4. Clase del dominio |
| 3 | Expresión regular, validación y limpieza | 3. Validar y limpiar · 4. Expresión regular |
| 4 | Reporte y archivos de salida | 3. Analizar y comunicar |
| 5 | Pruebas automáticas | 5. Verificar el comportamiento |
| 6 | Repositorio en GitHub | 6. Repositorio · 7. Documentación |

### Reglas de trabajo
- Los **nombres** de clases, métodos y funciones, y sus parámetros, son fijos: las pruebas automáticas los utilizan. Los cuerpos se escriben desde cero.
- Se utiliza únicamente la biblioteca estándar: `csv`, `json`, `re`, `abc` y `statistics`. Cada celda debe incluir los `import` que necesita.
- Distribución sugerida de tareas:
  - Integrante 1: Parte 1 (fuentes de datos).
  - Integrante 2: Partes 2 y 4 (clase `Paciente` y reporte).
  - Integrante 3: Parte 3 (validación y limpieza).
  - Integrante 4, si lo hay: en paralelo, crea el repositorio, el `README.md` y el `.gitignore` de la Parte 6.
- Todos los integrantes deben poder explicar el trabajo completo.

---
## Parte 0 — Datos de entrada (1 min)

Los datos provienen del archivo `EstudioPresionEdad.csv` del TP de Correlación y Regresión: datos **simulados con fines didácticos** de 50 pacientes adultos. Para esta actividad se adaptaron de la siguiente manera:
- cada paciente recibe un código con el formato `PAC-001` … `PAC-050`;
- se agregaron **8 registros con errores**, cada uno con un único problema;
- el mismo contenido se ofrece en dos formatos equivalentes: `mediciones.csv` y `mediciones.json`.

**Unidad de análisis:** cada registro corresponde a un paciente con una medición.

| Campo | Tipo | Descripción | Rango válido |
|---|---|---|---|
| `codigo` | texto | Identificador: `PAC-` seguido de tres dígitos | — |
| `edad` | número | Edad en años | 18 a 100 |
| `imc` | número | Índice de masa corporal (kg/m²) | 12 a 60 |
| `presion_sistolica` | número | Presión arterial sistólica (mmHg) | 70 a 250 |
| `grupo` | texto | `A` (control) o `B` (intervención) | `A`, `B` |

**Diferencia entre formatos que el programa debe resolver.** En el CSV **todos los valores se leen como texto** (`"21"`, `"28.8"`) y un dato faltante es la cadena vacía `""`. En el JSON los números se leen como números (`21`, `28.8`) y un dato faltante es `null`, que Python representa como `None`.

Ejecutar la celda siguiente para generar los dos archivos.

In [ ]:
# Celda provista: no modificar.
# Genera mediciones.csv y mediciones.json en la carpeta de trabajo.

DATOS_CSV = """codigo,edad,imc,presion_sistolica,grupo
PAC-001,21,28.8,97,B
PAC-002,22,30.1,107,A
PAC-003,25,27.2,101,A
PAC-004,26,25.7,112,A
PAC-005,26,22.8,116,A
PAC01,34,24.0,118,A
PAC-006,27,26.1,103,B
PAC-007,29,23.4,121,A
PAC-008,29,24.3,115,A
PAC-009,30,23.8,105,B
PAC-010,30,25.0,118,A
PAC-011,30,27.0,111,B
PAC-012,33,23.5,123,A
PAC-013,35,22.8,103,B
PAC-014,36,22.9,94,B
PAC-015,36,22.9,109,B
PAC-051,,25.3,120,B
PAC-016,37,22.1,104,B
PAC-017,39,23.3,111,B
PAC-018,40,27.8,126,B
PAC-019,41,24.1,114,A
PAC-020,41,25.8,127,A
PAC-052,48,26.0,999,A
PAC-021,44,23.1,128,A
PAC-022,45,25.9,120,B
PAC-023,45,22.1,121,B
PAC-024,46,24.3,113,B
PAC-025,46,30.2,117,B
PAC-026,46,24.5,131,A
PAC-027,49,28.2,134,A
PAC-028,49,26.8,130,B
PAC-029,54,26.0,132,A
PAC-030,55,25.1,141,A
PAC-012,33,23.5,180,A
PAC-031,56,29.1,133,A
PAC-032,56,26.1,117,B
PAC-033,56,29.5,122,A
PAC-034,56,25.0,124,A
PAC-035,57,26.7,139,A
PAC-053,39,abc,125,B
PAC-036,57,27.5,125,A
PAC-037,59,27.5,133,A
PAC-038,60,24.3,139,B
PAC-039,61,25.6,141,A
PAC-040,62,26.6,148,A
PAC-054,52,27.1,130,C
PAC-041,64,25.4,123,B
PAC-042,66,28.6,125,A
PAC-043,66,28.5,151,A
PAC-044,66,23.4,119,B
PAC-045,67,23.7,130,B
PAC-055,-45,24.8,122,A
PAC-046,67,24.8,140,A
PAC-047,68,27.7,135,A
PAC-048,70,31.7,149,B
PAC-049,71,26.9,145,B
PAC-050,74,28.4,142,B
PAC-056,61,28.0,,B
"""

DATOS_JSON = """[
  {"codigo": "PAC-001", "edad": 21, "imc": 28.8, "presion_sistolica": 97, "grupo": "B"},
  {"codigo": "PAC-002", "edad": 22, "imc": 30.1, "presion_sistolica": 107, "grupo": "A"},
  {"codigo": "PAC-003", "edad": 25, "imc": 27.2, "presion_sistolica": 101, "grupo": "A"},
  {"codigo": "PAC-004", "edad": 26, "imc": 25.7, "presion_sistolica": 112, "grupo": "A"},
  {"codigo": "PAC-005", "edad": 26, "imc": 22.8, "presion_sistolica": 116, "grupo": "A"},
  {"codigo": "PAC01", "edad": 34, "imc": 24.0, "presion_sistolica": 118, "grupo": "A"},
  {"codigo": "PAC-006", "edad": 27, "imc": 26.1, "presion_sistolica": 103, "grupo": "B"},
  {"codigo": "PAC-007", "edad": 29, "imc": 23.4, "presion_sistolica": 121, "grupo": "A"},
  {"codigo": "PAC-008", "edad": 29, "imc": 24.3, "presion_sistolica": 115, "grupo": "A"},
  {"codigo": "PAC-009", "edad": 30, "imc": 23.8, "presion_sistolica": 105, "grupo": "B"},
  {"codigo": "PAC-010", "edad": 30, "imc": 25.0, "presion_sistolica": 118, "grupo": "A"},
  {"codigo": "PAC-011", "edad": 30, "imc": 27.0, "presion_sistolica": 111, "grupo": "B"},
  {"codigo": "PAC-012", "edad": 33, "imc": 23.5, "presion_sistolica": 123, "grupo": "A"},
  {"codigo": "PAC-013", "edad": 35, "imc": 22.8, "presion_sistolica": 103, "grupo": "B"},
  {"codigo": "PAC-014", "edad": 36, "imc": 22.9, "presion_sistolica": 94, "grupo": "B"},
  {"codigo": "PAC-015", "edad": 36, "imc": 22.9, "presion_sistolica": 109, "grupo": "B"},
  {"codigo": "PAC-051", "edad": null, "imc": 25.3, "presion_sistolica": 120, "grupo": "B"},
  {"codigo": "PAC-016", "edad": 37, "imc": 22.1, "presion_sistolica": 104, "grupo": "B"},
  {"codigo": "PAC-017", "edad": 39, "imc": 23.3, "presion_sistolica": 111, "grupo": "B"},
  {"codigo": "PAC-018", "edad": 40, "imc": 27.8, "presion_sistolica": 126, "grupo": "B"},
  {"codigo": "PAC-019", "edad": 41, "imc": 24.1, "presion_sistolica": 114, "grupo": "A"},
  {"codigo": "PAC-020", "edad": 41, "imc": 25.8, "presion_sistolica": 127, "grupo": "A"},
  {"codigo": "PAC-052", "edad": 48, "imc": 26.0, "presion_sistolica": 999, "grupo": "A"},
  {"codigo": "PAC-021", "edad": 44, "imc": 23.1, "presion_sistolica": 128, "grupo": "A"},
  {"codigo": "PAC-022", "edad": 45, "imc": 25.9, "presion_sistolica": 120, "grupo": "B"},
  {"codigo": "PAC-023", "edad": 45, "imc": 22.1, "presion_sistolica": 121, "grupo": "B"},
  {"codigo": "PAC-024", "edad": 46, "imc": 24.3, "presion_sistolica": 113, "grupo": "B"},
  {"codigo": "PAC-025", "edad": 46, "imc": 30.2, "presion_sistolica": 117, "grupo": "B"},
  {"codigo": "PAC-026", "edad": 46, "imc": 24.5, "presion_sistolica": 131, "grupo": "A"},
  {"codigo": "PAC-027", "edad": 49, "imc": 28.2, "presion_sistolica": 134, "grupo": "A"},
  {"codigo": "PAC-028", "edad": 49, "imc": 26.8, "presion_sistolica": 130, "grupo": "B"},
  {"codigo": "PAC-029", "edad": 54, "imc": 26.0, "presion_sistolica": 132, "grupo": "A"},
  {"codigo": "PAC-030", "edad": 55, "imc": 25.1, "presion_sistolica": 141, "grupo": "A"},
  {"codigo": "PAC-012", "edad": 33, "imc": 23.5, "presion_sistolica": 180, "grupo": "A"},
  {"codigo": "PAC-031", "edad": 56, "imc": 29.1, "presion_sistolica": 133, "grupo": "A"},
  {"codigo": "PAC-032", "edad": 56, "imc": 26.1, "presion_sistolica": 117, "grupo": "B"},
  {"codigo": "PAC-033", "edad": 56, "imc": 29.5, "presion_sistolica": 122, "grupo": "A"},
  {"codigo": "PAC-034", "edad": 56, "imc": 25.0, "presion_sistolica": 124, "grupo": "A"},
  {"codigo": "PAC-035", "edad": 57, "imc": 26.7, "presion_sistolica": 139, "grupo": "A"},
  {"codigo": "PAC-053", "edad": 39, "imc": "abc", "presion_sistolica": 125, "grupo": "B"},
  {"codigo": "PAC-036", "edad": 57, "imc": 27.5, "presion_sistolica": 125, "grupo": "A"},
  {"codigo": "PAC-037", "edad": 59, "imc": 27.5, "presion_sistolica": 133, "grupo": "A"},
  {"codigo": "PAC-038", "edad": 60, "imc": 24.3, "presion_sistolica": 139, "grupo": "B"},
  {"codigo": "PAC-039", "edad": 61, "imc": 25.6, "presion_sistolica": 141, "grupo": "A"},
  {"codigo": "PAC-040", "edad": 62, "imc": 26.6, "presion_sistolica": 148, "grupo": "A"},
  {"codigo": "PAC-054", "edad": 52, "imc": 27.1, "presion_sistolica": 130, "grupo": "C"},
  {"codigo": "PAC-041", "edad": 64, "imc": 25.4, "presion_sistolica": 123, "grupo": "B"},
  {"codigo": "PAC-042", "edad": 66, "imc": 28.6, "presion_sistolica": 125, "grupo": "A"},
  {"codigo": "PAC-043", "edad": 66, "imc": 28.5, "presion_sistolica": 151, "grupo": "A"},
  {"codigo": "PAC-044", "edad": 66, "imc": 23.4, "presion_sistolica": 119, "grupo": "B"},
  {"codigo": "PAC-045", "edad": 67, "imc": 23.7, "presion_sistolica": 130, "grupo": "B"},
  {"codigo": "PAC-055", "edad": -45, "imc": 24.8, "presion_sistolica": 122, "grupo": "A"},
  {"codigo": "PAC-046", "edad": 67, "imc": 24.8, "presion_sistolica": 140, "grupo": "A"},
  {"codigo": "PAC-047", "edad": 68, "imc": 27.7, "presion_sistolica": 135, "grupo": "A"},
  {"codigo": "PAC-048", "edad": 70, "imc": 31.7, "presion_sistolica": 149, "grupo": "B"},
  {"codigo": "PAC-049", "edad": 71, "imc": 26.9, "presion_sistolica": 145, "grupo": "B"},
  {"codigo": "PAC-050", "edad": 74, "imc": 28.4, "presion_sistolica": 142, "grupo": "B"},
  {"codigo": "PAC-056", "edad": 61, "imc": 28.0, "presion_sistolica": null, "grupo": "B"}
]
"""

with open("mediciones.csv", "w", encoding="utf-8", newline="") as archivo:
    archivo.write(DATOS_CSV)
with open("mediciones.json", "w", encoding="utf-8") as archivo:
    archivo.write(DATOS_JSON)

print("Archivos generados: mediciones.csv y mediciones.json")

In [ ]:
# Celda provista: primeras líneas de cada archivo
with open("mediciones.csv", encoding="utf-8") as archivo:
    print("".join(archivo.readlines()[:4]))
with open("mediciones.json", encoding="utf-8") as archivo:
    print("".join(archivo.readlines()[:4]))

---
## Parte 1 — Fuentes de datos (8 min)

El resto del programa no debe depender del formato del archivo. Para ello se define una **interfaz común**: toda fuente de datos implementa `cargar(ruta)` y devuelve el mismo tipo de resultado, una **lista de diccionarios** (un diccionario por registro, con los nombres de los campos como claves).

Implementar:
- `FuenteDatos`: clase **abstracta** (hereda de `ABC`) con el método abstracto `cargar(self, ruta)`.
- `FuenteCSV(FuenteDatos)`: `cargar` lee el archivo con `csv.DictReader` y devuelve la lista de diccionarios.
- `FuenteJSON(FuenteDatos)`: `cargar` lee el archivo con `json.load`. Si el contenido **no es una lista**, lanza `ValueError` con un mensaje que indique el problema.

Condiciones:
- Abrir los archivos con `encoding="utf-8"` (en el caso del CSV, también con `newline=""`).
- **No capturar** el `FileNotFoundError` cuando el archivo no existe: una falla de lectura no debe ocultarse.
- En esta etapa no se convierten tipos: cada fuente devuelve los valores tal como los entrega su formato.

In [ ]:
# Escribir aquí los import necesarios y las clases FuenteDatos, FuenteCSV y FuenteJSON

In [ ]:
# Verificación rápida
registros_csv = FuenteCSV().cargar("mediciones.csv")
registros_json = FuenteJSON().cargar("mediciones.json")
print(len(registros_csv), len(registros_json))  # 58 58
print(registros_csv[0])   # {'codigo': 'PAC-001', 'edad': '21', 'imc': '28.8', ...}
print(registros_json[0])  # {'codigo': 'PAC-001', 'edad': 21, 'imc': 28.8, ...}

---
## Parte 2 — Clase `Paciente` (7 min)

`Paciente` representa un registro **ya validado**: sus atributos numéricos son `float` y su grupo es `"A"` o `"B"`. Se construye con `Paciente(codigo, edad, imc, presion, grupo)`.

| Miembro | Descripción |
|---|---|
| atributos `codigo`, `edad`, `imc`, `presion`, `grupo` | Los valores recibidos en el constructor |
| `franja_etaria()` | Devuelve `"18-39"` si la edad es menor que 40, `"40-59"` si es menor que 60 y `"60+"` en otro caso |
| `es_hipertenso()` | Devuelve `True` si la presión sistólica es mayor o igual que el umbral de 140 mmHg |
| `__str__()` | Devuelve un texto con el código, el grupo, la edad y la presión (ver el ejemplo siguiente) |

Ejemplo de la representación de texto: `PAC-001 | grupo B | 21 años | PAS 97 mmHg`

El umbral de 140 mmHg no debe figurar como un número aislado dentro del método: se recomienda definirlo como **atributo de clase** con un nombre descriptivo.

> **Nota:** el umbral es un criterio didáctico simplificado y no constituye un diagnóstico clínico.

In [ ]:
# Escribir aquí la clase Paciente

In [ ]:
p = Paciente("PAC-001", 21.0, 28.8, 97.0, "B")
print(p)                  # PAC-001 | grupo B | 21 años | PAS 97 mmHg
print(p.franja_etaria())  # 18-39
print(p.es_hipertenso())  # False

---
## Parte 3 — Validación y limpieza (13 min)

### 3.a Expresión regular para el código (provista)

Una **expresión regular** describe un patrón de texto. El módulo `re` permite comprobar si una cadena respeta ese patrón. El código de paciente debe estar formado por `PAC-` seguido de exactamente tres dígitos:

```python
PATRON_CODIGO = re.compile(r"PAC-\d{3}")
```

| Fragmento | Significado |
|---|---|
| `r"..."` | Cadena *raw*: Python no interpreta la barra invertida `\`, que queda disponible para la expresión regular |
| `PAC-` | Caracteres literales: deben aparecer exactamente así, en mayúsculas |
| `\d` | Un dígito cualquiera, de `0` a `9` |
| `{3}` | El elemento anterior se repite exactamente 3 veces |

`re.compile` construye el patrón una sola vez para reutilizarlo. Para validar se utiliza el método `fullmatch(texto)`, que exige que **todo** el texto respete el patrón: devuelve un objeto de coincidencia si lo respeta y `None` en caso contrario. El método `search`, en cambio, busca el patrón en cualquier parte del texto y aceptaría, por ejemplo, `"XPAC-0012"`.

La expresión regular comprueba **solo el formato**: `PAC-000` tiene un formato válido aunque ningún paciente tenga ese número. Las reglas que no son de formato, como los rangos o los duplicados, requieren comprobaciones adicionales.

Ejecutar la celda siguiente para observar el comportamiento del patrón.

In [ ]:
import re

PATRON_CODIGO = re.compile(r"PAC-\d{3}")

for texto in ["PAC-001", "PAC01", "PAC-0001", "pac-001", "PAC-00A", "XPAC-0012"]:
    respeta_todo = PATRON_CODIGO.fullmatch(texto) is not None
    aparece = PATRON_CODIGO.search(texto) is not None
    print(f"{texto:<10} fullmatch: {respeta_todo!s:<6} search: {aparece}")

### 3.b Funciones de validación y limpieza

Implementar las tres funciones siguientes. Se recomienda definir los rangos válidos y los grupos admitidos como **constantes** al comienzo de la celda (por ejemplo, un diccionario `RANGOS = {"edad": (18, 100), ...}`), en lugar de repetir los valores dentro de las funciones.

**`convertir_numero(valor)`**
- Si `valor` es `None` o una cadena vacía (o formada solo por espacios), devuelve `None`: el dato **falta**.
- En otro caso devuelve `float(valor)`, que funciona tanto para `"28.8"` (CSV) como para `28.8` (JSON).
- Si el valor no representa un número (por ejemplo, `"abc"`), deja que `float` lance `ValueError`.

**`validar_registro(registro)`** → lista de motivos (cadenas de texto)
- Devuelve una lista **vacía** si el registro es válido, o una lista con **un motivo por cada problema** encontrado, por ejemplo `["edad faltante"]` o `["código con formato inválido"]`.
- Comprueba: el formato del código con `PATRON_CODIGO.fullmatch`; que `edad`, `imc` y `presion_sistolica` sean numéricos, no falten y estén dentro de su rango; que `grupo` sea `"A"` o `"B"`.
- No comprueba duplicados, porque eso requiere conocer los registros anteriores.

**`limpiar(registros)`** → tupla `(validos, rechazos)`
- Recorre los registros en orden. Cada registro sin motivos de rechazo, y cuyo código no haya sido aceptado antes, se convierte en un objeto `Paciente` con valores `float` y se agrega a `validos`.
- **Regla de duplicados:** si el código ya fue aceptado, el registro se rechaza con el motivo `"código duplicado"`. **Se conserva la primera aparición.**
- Cada rechazo se registra como un diccionario `{"fila": n, "codigo": ..., "motivo": ...}`, donde `n` es la posición del registro en la lista **contando desde 1** y `motivo` reúne los motivos separados por `"; "`.
- No modifica la lista recibida.

In [ ]:
# PATRON_CODIGO ya está definido en la celda anterior.
# Escribir aquí las constantes y las funciones convertir_numero, validar_registro y limpiar

In [ ]:
print(convertir_numero("28.8"), convertir_numero(28.8), convertir_numero(""), convertir_numero(None))
print(validar_registro({"codigo": "PAC-001", "edad": "21", "imc": "28.8", "presion_sistolica": "97", "grupo": "B"}))
print(validar_registro({"codigo": "PAC01", "edad": "", "imc": "abc", "presion_sistolica": "999", "grupo": "C"}))

validos, rechazos = limpiar(FuenteCSV().cargar("mediciones.csv"))
print(len(validos), "válidos,", len(rechazos), "rechazados")  # 50 válidos, 8 rechazados
for rechazo in rechazos:
    print(rechazo)

---
## Parte 4 — Reporte y archivos de salida (8 min)

**`generar_reporte(pacientes)`** → lista de diccionarios con **una fila por grupo**, ordenada por grupo:

| Clave | Contenido |
|---|---|
| `grupo` | `"A"` o `"B"` |
| `cantidad` | Cantidad de pacientes del grupo |
| `presion_media` | Promedio de la presión sistólica, redondeado a 2 decimales |
| `proporcion_hipertensos` | Fracción de pacientes del grupo para los que `es_hipertenso()` es verdadero, redondeada a 3 decimales |

Si la lista de pacientes está vacía, la función no debe producir una división por cero.

**`guardar_csv(filas, ruta)`**: guarda una lista de diccionarios en un archivo CSV mediante `csv.DictWriter`; los encabezados son las claves del primer diccionario. Puede suponerse que la lista no está vacía. Abrir el archivo con `newline=""` y `encoding="utf-8"`.

In [ ]:
# Escribir aquí las funciones generar_reporte y guardar_csv

In [ ]:
reporte = generar_reporte(validos)
for fila in reporte:
    print(fila)

guardar_csv(reporte, "reporte.csv")
guardar_csv(rechazos, "rechazos.csv")
print(FuenteCSV().cargar("reporte.csv"))

---
## Parte 5 — Pruebas automáticas (3 min)

Ejecutar la celda siguiente. Si todo es correcto, se imprime **`Todas las pruebas pasaron.`** Si aparece un `AssertionError`, el mensaje indica qué comportamiento falla.

Las pruebas cubren los tres tipos de verificación que exige el TP2:
1. **Caso normal:** lectura de ambos formatos, 50 registros válidos y 8 rechazados.
2. **Entradas inválidas y casos límite:** códigos mal formados, archivo inexistente, JSON con estructura incorrecta, edades en el límite de cada franja y lista vacía.
3. **Propiedades del resultado:** ningún registro se pierde (válidos + rechazados = leídos), las lecturas CSV y JSON producen el mismo reporte y los resultados coinciden con los **valores de control del TP de EDA**: correlación entre edad y presión sistólica r = 0,794 y medias por grupo. Esta última verificación se cumple solo si la limpieza recuperó exactamente los 50 registros originales.

In [ ]:
import json
import statistics  # statistics.correlation requiere Python 3.10 o superior
import tempfile
from pathlib import Path

# --- Configuración (en tests/test_pipeline.py se reemplaza por el encabezado de la Parte 6) ---
RUTA_CSV = "mediciones.csv"
RUTA_JSON = "mediciones.json"


def probar(condicion, mensaje):
    assert condicion, mensaje


def debe_fallar(funcion, mensaje, error=ValueError):
    try:
        funcion()
    except error:
        return
    probar(False, mensaje)


carpeta_temporal = Path(tempfile.mkdtemp())

# --- 1. Caso normal ---
registros_csv = FuenteCSV().cargar(RUTA_CSV)
registros_json = FuenteJSON().cargar(RUTA_JSON)
probar(len(registros_csv) == 58, "FuenteCSV debe leer los 58 registros")
probar(len(registros_json) == 58, "FuenteJSON debe leer los 58 registros")
probar(isinstance(registros_csv[0], dict), "cargar() debe devolver una lista de diccionarios")
probar(set(registros_csv[0]) == {"codigo", "edad", "imc", "presion_sistolica", "grupo"},
       "Las claves de cada registro deben ser los nombres de los campos")

validos, rechazos = limpiar(registros_csv)
probar(len(validos) == 50, f"Se esperaban 50 registros válidos y se obtuvieron {len(validos)}")
probar(len(rechazos) == 8, f"Se esperaban 8 rechazos y se obtuvieron {len(rechazos)}")
probar(all(isinstance(p, Paciente) for p in validos), "limpiar() debe devolver objetos Paciente")
probar(all(isinstance(p.edad, float) and isinstance(p.presion, float) for p in validos),
       "Los atributos numéricos de Paciente deben ser float")
probar(all(r["motivo"] for r in rechazos), "Cada rechazo debe indicar un motivo")
probar(validar_registro({"codigo": "PAC-001", "edad": "21", "imc": "28.8",
                         "presion_sistolica": "97", "grupo": "B"}) == [],
       "Un registro correcto no debe tener motivos de rechazo")

# --- 2. Entradas inválidas y casos límite ---
probar(convertir_numero("28.8") == 28.8 and convertir_numero(28.8) == 28.8,
       "convertir_numero debe aceptar texto y números")
probar(convertir_numero("") is None and convertir_numero(None) is None,
       "Un valor faltante debe convertirse en None")
debe_fallar(lambda: convertir_numero("abc"), "convertir_numero('abc') debe lanzar ValueError")

base = {"codigo": "PAC-001", "edad": "40", "imc": "25", "presion_sistolica": "120", "grupo": "A"}
for codigo in ["PAC01", "PAC-0001", "pac-001", "XPAC-001"]:
    probar(validar_registro({**base, "codigo": codigo}) != [], f"El código {codigo} debe rechazarse")
probar(validar_registro({**base, "edad": "-45"}) != [], "Una edad negativa debe rechazarse")
probar(validar_registro({**base, "presion_sistolica": None}) != [], "Una presión faltante debe rechazarse")
probar(validar_registro({**base, "grupo": "C"}) != [], "Un grupo distinto de A y B debe rechazarse")
probar(validar_registro({**base, "imc": "abc"}) != [], "Un IMC no numérico debe rechazarse")

codigos_rechazados = [r["codigo"] for r in rechazos]
probar("PAC01" in codigos_rechazados, "El código PAC01 debe figurar entre los rechazos")
probar([r["fila"] for r in rechazos if r["codigo"] == "PAC01"] == [6],
       "El rechazo de PAC01 debe indicar la fila 6 (contando desde 1)")
probar("PAC-012" in codigos_rechazados, "La segunda aparición de PAC-012 debe rechazarse por duplicada")
paciente_12 = [p for p in validos if p.codigo == "PAC-012"]
probar(len(paciente_12) == 1 and paciente_12[0].presion == 123.0,
       "Ante un código duplicado debe conservarse la primera aparición")

debe_fallar(lambda: FuenteCSV().cargar("no_existe.csv"),
            "Un archivo inexistente debe lanzar FileNotFoundError", FileNotFoundError)
ruta_no_lista = carpeta_temporal / "no_lista.json"
ruta_no_lista.write_text(json.dumps({"codigo": "PAC-001"}), encoding="utf-8")
debe_fallar(lambda: FuenteJSON().cargar(ruta_no_lista),
            "FuenteJSON debe lanzar ValueError si el contenido no es una lista")
debe_fallar(lambda: FuenteDatos(), "FuenteDatos debe ser abstracta", TypeError)

for edad, franja in [(18, "18-39"), (39, "18-39"), (40, "40-59"), (59, "40-59"), (60, "60+")]:
    probar(Paciente("PAC-001", float(edad), 25.0, 120.0, "A").franja_etaria() == franja,
           f"La edad {edad} corresponde a la franja {franja}")
probar(Paciente("PAC-001", 50.0, 25.0, 140.0, "A").es_hipertenso(), "PAS = 140 debe considerarse hipertensión")
probar(not Paciente("PAC-001", 50.0, 25.0, 139.0, "A").es_hipertenso(), "PAS = 139 no es hipertensión")
probar("PAC-001" in str(Paciente("PAC-001", 50.0, 25.0, 120.0, "A")), "__str__ debe mostrar el código")

try:
    reporte_vacio = generar_reporte([])
except ZeroDivisionError:
    probar(False, "generar_reporte([]) no debe dividir por cero")
probar(all(fila["cantidad"] == 0 for fila in reporte_vacio), "Un reporte vacío no debe contar pacientes")

# --- 3. Propiedades del resultado ---
probar(len(validos) + len(rechazos) == len(registros_csv),
       "Ningún registro puede perderse: válidos + rechazados = leídos")
probar(len({p.codigo for p in validos}) == len(validos), "No debe haber códigos repetidos entre los válidos")

validos_json, rechazos_json = limpiar(registros_json)
reporte_csv = generar_reporte(validos)
probar(generar_reporte(validos_json) == reporte_csv, "Las lecturas CSV y JSON deben producir el mismo reporte")
probar([r["motivo"] for r in rechazos_json] == [r["motivo"] for r in rechazos],
       "Las lecturas CSV y JSON deben producir los mismos rechazos")

probar([fila["grupo"] for fila in reporte_csv] == ["A", "B"], "El reporte debe tener una fila por grupo, ordenadas")
fila_a, fila_b = reporte_csv
probar(fila_a["cantidad"] == 27 and fila_b["cantidad"] == 23, "El grupo A debe tener 27 pacientes y el B, 23")
probar(abs(fila_a["presion_media"] - 127.26) < 0.01, "La PAS media del grupo A debe ser 127.26")
probar(abs(fila_b["presion_media"] - 118.61) < 0.01, "La PAS media del grupo B debe ser 118.61")
probar(abs(fila_a["proporcion_hipertensos"] - 5 / 27) < 0.001, "La proporción de hipertensos del grupo A debe ser 5/27")

r = statistics.correlation([p.edad for p in validos], [p.presion for p in validos])
probar(abs(r - 0.794) < 0.001, f"r(edad, PAS) debe ser 0.794 (Anexo A del TP de EDA) y se obtuvo {r:.4f}")

ruta_reporte = carpeta_temporal / "reporte.csv"
guardar_csv(reporte_csv, ruta_reporte)
releido = FuenteCSV().cargar(ruta_reporte)
probar(len(releido) == 2 and releido[0]["grupo"] == "A", "guardar_csv debe escribir un CSV legible por FuenteCSV")

print("Todas las pruebas pasaron.")

### Flujo completo

La celda siguiente reúne todos los pasos en una función `main`. Su contenido es el que tendrá `src/main.py` en el repositorio.

In [ ]:
from pathlib import Path

CARPETA_SALIDA = Path("outputs")


def main(ruta_entrada):
    registros = FuenteCSV().cargar(ruta_entrada)
    validos, rechazos = limpiar(registros)
    reporte = generar_reporte(validos)

    print(f"Leídos: {len(registros)} | válidos: {len(validos)} | rechazados: {len(rechazos)}")
    for fila in reporte:
        print(fila)

    CARPETA_SALIDA.mkdir(exist_ok=True)
    guardar_csv(reporte, CARPETA_SALIDA / "reporte.csv")
    if rechazos:
        guardar_csv(rechazos, CARPETA_SALIDA / "rechazos.csv")


main("mediciones.csv")

---
## Parte 6 — Repositorio en GitHub (15 min)

Un integrante crea el repositorio **público** `tp-pipeline-presion-grupoX`, agrega al resto como colaboradores y se organiza el código con la siguiente estructura, versión reducida de la que exige el TP2:

```
tp-pipeline-presion-grupoX/
├── README.md
├── .gitignore
├── data/
│   ├── README.md          ← procedencia y descripción de los datos
│   ├── mediciones.csv
│   └── mediciones.json
├── src/
│   ├── fuentes.py         ← FuenteDatos, FuenteCSV, FuenteJSON
│   ├── modelo.py          ← Paciente
│   ├── pipeline.py        ← PATRON_CODIGO, constantes y funciones de las Partes 3 y 4
│   └── main.py            ← punto de entrada del programa
└── tests/
    └── test_pipeline.py   ← pruebas de la Parte 5
```

Cada archivo `.py` incluye los `import` que necesita: `pipeline.py` utiliza `import csv`, `import re` y `from modelo import Paciente`; `main.py` importa desde `fuentes` y `pipeline`.

> **Sugerencia para Colab:** una celda que comienza con `%%writefile src/fuentes.py` guarda su contenido en ese archivo. Las carpetas se crean previamente con `!mkdir -p src tests data`. Luego los archivos se descargan desde el panel lateral.

### `src/main.py`
```python
"""Punto de entrada. Uso: python src/main.py data/mediciones.csv"""
import sys
from pathlib import Path

from fuentes import FuenteCSV
from pipeline import limpiar, generar_reporte, guardar_csv

CARPETA_SALIDA = Path("outputs")


def main(ruta_entrada):
    registros = FuenteCSV().cargar(ruta_entrada)
    validos, rechazos = limpiar(registros)
    reporte = generar_reporte(validos)

    print(f"Leídos: {len(registros)} | válidos: {len(validos)} | rechazados: {len(rechazos)}")
    for fila in reporte:
        print(fila)

    CARPETA_SALIDA.mkdir(exist_ok=True)
    guardar_csv(reporte, CARPETA_SALIDA / "reporte.csv")
    if rechazos:
        guardar_csv(rechazos, CARPETA_SALIDA / "rechazos.csv")


if __name__ == "__main__":
    if len(sys.argv) != 2:
        sys.exit("Uso: python src/main.py <archivo_de_mediciones.csv>")
    main(sys.argv[1])
```

### `tests/test_pipeline.py`
Copiar la celda de la Parte 5 y reemplazar el bloque de configuración (las líneas de `RUTA_CSV` y `RUTA_JSON`) por el siguiente encabezado:
```python
import sys

RAIZ = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(RAIZ / "src"))  # permite importar los módulos de src/

from fuentes import FuenteDatos, FuenteCSV, FuenteJSON
from modelo import Paciente
from pipeline import convertir_numero, validar_registro, limpiar, generar_reporte, guardar_csv

RUTA_CSV = RAIZ / "data" / "mediciones.csv"
RUTA_JSON = RAIZ / "data" / "mediciones.json"
```

### `.gitignore`
```
__pycache__/
.venv/
.ipynb_checkpoints/
outputs/
```

### Plantilla de `README.md`
~~~markdown
# Pipeline de control de presión arterial

**Integrantes:** Apellido, Nombre · Apellido, Nombre · Apellido, Nombre

## Problema y usuario
Un equipo de salud recibe mediciones de presión arterial en CSV y JSON con errores de carga.
El programa descarta los registros inválidos, informa el motivo de cada rechazo y resume
la presión sistólica por grupo del estudio.

## Datos
Ver `data/README.md`. Cada fila corresponde a un paciente con una medición.

## Ejecución
No requiere bibliotecas externas (Python 3.10 o superior).
```bash
python src/main.py data/mediciones.csv
```

## Pruebas
```bash
python tests/test_pipeline.py
```

## Reglas de limpieza
- Código con formato `PAC-` y tres dígitos (expresión regular `PAC-\d{3}`).
- (completar: rangos válidos, grupos admitidos, tratamiento de faltantes y duplicados)

## Resultado de ejemplo
(pegar la salida del programa)

## Limitaciones
(completar)
~~~

### Plantilla de `data/README.md`
~~~markdown
# Datos
- **Fuente:** datos simulados con fines didácticos, provistos por la cátedra
  (TP de Correlación y Regresión, archivo `EstudioPresionEdad.csv`).
- **Adaptación:** se agregó un código por paciente y 8 registros con errores para ejercitar la validación.
- **Unidad de análisis:** cada fila es un paciente adulto con una medición.
- **Variables:** codigo, edad (años), imc (kg/m²), presion_sistolica (mmHg), grupo (A: control, B: intervención).
- **Limitaciones:** datos simulados; 50 pacientes; criterio de hipertensión simplificado.
~~~

### Commits
```bash
git add .
git commit -m "Agrega fuentes CSV y JSON"
git push
```
El historial debe tener **al menos 3 commits** con mensajes que describan cambios concretos, por ejemplo: "Agrega fuentes CSV y JSON", "Agrega validación y limpieza", "Agrega reporte y pruebas". No se admiten commits vacíos o artificiales.

> Si no es posible utilizar la terminal, los archivos pueden subirse desde la web: **Add file → Upload files → Commit changes**. Cada subida genera un commit.

### Criterio de finalización
Desde la carpeta raíz del repositorio:
- `python src/main.py data/mediciones.csv` imprime el resumen y genera `outputs/reporte.csv` y `outputs/rechazos.csv`;
- `python tests/test_pipeline.py` imprime `Todas las pruebas pasaron.`

Esta estructura coincide con lo que solicita el **hito de Avance del TP2** (repositorio creado, README inicial, estructura y primer flujo de lectura y validación), por lo que puede tomarse como modelo para el proyecto individual.

### Entrega
Copiar el enlace del repositorio y entregarlo en el aula virtual o en el formulario que indique la cátedra.

---
### Desafíos adicionales (si el tiempo lo permite)
- Implementar `reporte_por_franja(pacientes)`, que agrupe por `franja_etaria()` en lugar de por grupo.
- Modificar `main.py` para que elija la fuente según la extensión del archivo (`.csv` → `FuenteCSV`, `.json` → `FuenteJSON`) y lance un error claro si la extensión no corresponde a ninguna de las dos.
- Agregar un campo con la fecha de medición en formato `AAAA-MM-DD`. El patrón `\d{4}-\d{2}-\d{2}` acepta `2026-02-30`, una fecha inexistente. ¿Qué comprobación complementaria se necesita? (Sugerencia: `datetime.strptime`.)